# 춘천 닭갈비 데이터로 선형회귀 확장 실습하기

이 노트북은 GitHub에 올려 둔 CSV 파일을 바로 읽어와 단순 선형회귀와 다중 선형회귀를 실습합니다.

- 단순 선형회귀: 독립변수 X 1개, 종속변수 Y 1개
- 다중 선형회귀: 독립변수 X 2개, 종속변수 Y 1개, 3D 산점도와 회귀 평면
- 다중 선형회귀: 독립변수 X N개, 종속변수 Y 1개, 체크박스 선택과 X1 기준 2D 시각화
- 변수 선택은 수치형 데이터로 제한
- 시각화는 Plotly 사용

## 1. 라이브러리 준비
Colab에서 위젯과 Plotly 그래프가 안정적으로 실행되도록 필요한 라이브러리를 준비합니다.

In [ ]:
# Google Colab 실행 준비
!pip -q install plotly ipywidgets scikit-learn pandas

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

In [ ]:
from __future__ import annotations

import io
import math
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from IPython.display import display, HTML, clear_output
from ipywidgets import Button, Checkbox, Dropdown, GridBox, HBox, Layout, Output, VBox
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

PLOTLY_TEMPLATE = "plotly_white"
ENCODINGS = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]


def clean_column_names(columns):
    seen = {}
    result = []
    for col in columns:
        base = str(col).replace("\ufeff", "").strip()
        if base == "":
            base = "이름없는_컬럼"
        if base not in seen:
            seen[base] = 0
            result.append(base)
        else:
            seen[base] += 1
            result.append(f"{base}_{seen[base]}")
    return result


def read_csv_with_fallback(source):
    last_error = None
    for encoding in ENCODINGS:
        try:
            dataframe = pd.read_csv(source, encoding=encoding)
            dataframe.columns = clean_column_names(dataframe.columns)
            return dataframe, encoding
        except Exception as error:
            last_error = error
    raise ValueError(f"CSV 파일을 읽지 못했습니다. 마지막 오류: {last_error}")


def extract_uploaded_file(upload_widget):
    value = upload_widget.value
    if not value:
        return None, None

    if isinstance(value, dict):
        file_name, info = next(iter(value.items()))
        content = info.get("content", b"")
    else:
        info = value[0]
        file_name = info.get("name", "uploaded.csv")
        content = info.get("content", b"")

    if hasattr(content, "tobytes"):
        content = content.tobytes()
    if isinstance(content, memoryview):
        content = content.tobytes()

    return file_name, bytes(content)


def prepare_numeric_dataframe(dataframe):
    numeric_df = dataframe.copy()
    for col in numeric_df.columns:
        if pd.api.types.is_numeric_dtype(numeric_df[col]):
            continue
        converted = pd.to_numeric(
            numeric_df[col].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )
        original_notna = numeric_df[col].notna().sum()
        converted_notna = converted.notna().sum()
        if original_notna > 0 and converted_notna == original_notna:
            numeric_df[col] = converted

    numeric_cols = numeric_df.select_dtypes(include=[np.number]).columns.tolist()
    return numeric_df, numeric_cols


def show_df_info(dataframe):
    buffer = io.StringIO()
    dataframe.info(buf=buffer)
    display(HTML("<h4>df.info()</h4>"))
    display(HTML(f"<pre>{buffer.getvalue()}</pre>"))


def column_profile(dataframe):
    rows = []
    for col in dataframe.columns:
        series = dataframe[col]
        row = {
            "컬럼명": col,
            "자료형": str(series.dtype),
            "결측치": int(series.isna().sum()),
            "고유값 수": int(series.nunique(dropna=True)),
        }
        if pd.api.types.is_numeric_dtype(series):
            row.update(
                {
                    "최솟값": round(float(series.min()), 4) if series.notna().any() else np.nan,
                    "최댓값": round(float(series.max()), 4) if series.notna().any() else np.nan,
                    "평균": round(float(series.mean()), 4) if series.notna().any() else np.nan,
                }
            )
        else:
            row.update({"최솟값": "", "최댓값": "", "평균": ""})
        rows.append(row)
    return pd.DataFrame(rows)


def display_numeric_columns(dataframe):
    numeric_df, numeric_cols = prepare_numeric_dataframe(dataframe)
    display(HTML("<h4>회귀 분석에 사용할 수 있는 수치형 변수</h4>"))
    if numeric_cols:
        display(column_profile(numeric_df[numeric_cols]))
    else:
        display(HTML("<b>수치형 변수로 해석할 수 있는 컬럼이 없습니다.</b>"))
    return numeric_df, numeric_cols

## 2. GitHub CSV 읽기
아래 셀을 실행하면 GitHub raw URL의 CSV가 `df`라는 pandas DataFrame으로 바로 읽힙니다.

In [ ]:
CSV_URL = "https://raw.githubusercontent.com/CarlosQuperman/AIEDAP2026_LOCAL_DATA/main/5%EC%9E%A5/%EC%B6%98%EC%B2%9C_%EB%8B%AD%EA%B0%88%EB%B9%84_%EC%83%81%EC%A0%90%EB%B3%84_%EC%B5%9C%EA%B3%A0_%EC%9D%B4%EC%9A%A9%EC%9E%90%EC%88%98.csv"

df, encoding_used = read_csv_with_fallback(CSV_URL)
display(HTML(f"<h4>데이터 읽기 완료</h4><p>인코딩: <code>{encoding_used}</code>, 행: {len(df)}, 열: {len(df.columns)}</p>"))
display(df.head())

## 3. 데이터 기본 정보 확인
데이터의 행과 열, 결측치, 자료형을 확인합니다. 회귀 변수 선택에는 수치형 컬럼만 사용합니다.

In [ ]:
show_df_info(df)
display(column_profile(df))
numeric_df, numeric_cols = display_numeric_columns(df)

## 4. 회귀 결과 읽는 법

회귀 분석은 실제값과 예측값의 차이를 보며 모델이 데이터를 얼마나 잘 설명하는지 확인합니다. 아래 용어는 모델을 평가할 때 자주 사용됩니다.

- **R²(결정계수)**: 실제 Y 값의 변화 중 모델이 설명하는 비율입니다. 예를 들어 `R² = 0.72`라면 이 데이터에서 Y의 변화 약 72%를 선택한 X 변수들이 설명한다고 볼 수 있습니다. 다만 R²가 높다고 원인과 결과가 증명되는 것은 아닙니다.
- **MAE(평균 절대 오차)**: `|실제값 - 예측값|`의 평균입니다. 예를 들어 이용자수를 예측할 때 `MAE = 120`이면 예측이 평균적으로 약 120명 정도 빗나간다는 뜻입니다.
- **RMSE(평균 제곱근 오차)**: 큰 오차에 더 민감한 평균 오차입니다. 예를 들어 `MAE = 120`, `RMSE = 250`이라면 일부 상점에서 예측이 크게 빗나갔을 가능성을 생각해 볼 수 있습니다.
- **잔차**: `실제값 - 예측값`입니다. 실제 이용자수가 1000명이고 예측값이 850명이면 잔차는 `150`입니다. 잔차가 0 주변에 고르게 흩어질수록 선형 모델이 비교적 자연스럽게 데이터를 설명한다고 볼 수 있습니다.

## 5. 변수 직접 선택 후 선형회귀 실행
회귀 유형을 고르고 X와 Y를 직접 선택한 뒤 실행합니다. N개 입력 다중 회귀에서는 X 변수를 체크박스로 선택합니다.

In [ ]:
class ExtendedLinearRegressionColabApp:
    def __init__(self, dataframe, dataset_name="데이터"):
        self.original_df = dataframe.copy()
        self.dataset_name = dataset_name
        self.numeric_df, self.numeric_cols = prepare_numeric_dataframe(self.original_df)
        self.output = Output()

        self.latest_mode = None
        self.latest_x_cols = None
        self.latest_y_col = None
        self.latest_plot_x = None
        self.latest_model = None
        self.latest_result = None

        self.mode_dropdown = Dropdown(
            options=[
                ("단순 선형회귀: 입력값 1개, 출력값 1개", "simple"),
                ("다중 선형회귀: 입력값 2개, 출력값 1개", "two"),
                ("다중 선형회귀: 입력값 N개, 출력값 1개", "n"),
            ],
            value="simple",
            description="회귀 유형",
            layout={"width": "520px"},
        )
        self.x1_dropdown = Dropdown(options=self.numeric_cols, description="X1", layout={"width": "330px"})
        self.x2_dropdown = Dropdown(options=self.numeric_cols, description="X2", layout={"width": "330px"})
        self.y_dropdown = Dropdown(options=self.numeric_cols, description="Y", layout={"width": "330px"})
        self.plot_x_dropdown = Dropdown(
            options=self.numeric_cols,
            description="시각화 X1",
            layout={"width": "360px"},
        )

        self.x_checkboxes = [
            Checkbox(value=False, description=col, indent=False, layout=Layout(width="230px"))
            for col in self.numeric_cols
        ]
        for checkbox in self.x_checkboxes:
            checkbox.observe(self.update_checked_x_options, names="value")

        self.checkbox_grid = GridBox(
            self.x_checkboxes,
            layout=Layout(
                grid_template_columns="repeat(2, 240px)",
                grid_gap="4px 12px",
                max_height="260px",
                overflow_y="auto",
                border="1px solid #ddd",
                padding="8px",
            ),
        )
        self.n_box = VBox([self.checkbox_grid, self.plot_x_dropdown])
        self.run_button = Button(description="회귀 분석 실행", button_style="success", icon="play")

        self.mode_dropdown.observe(self.update_mode, names="value")
        self.y_dropdown.observe(self.update_checked_x_options, names="value")
        self.run_button.on_click(self.run_regression)
        self.update_mode()

    def selected_checked_x(self):
        return [checkbox.description for checkbox in self.x_checkboxes if checkbox.value]

    def update_checked_x_options(self, change=None):
        checked = self.selected_checked_x()
        current = self.plot_x_dropdown.value
        self.plot_x_dropdown.options = checked if checked else self.numeric_cols
        if checked and current in checked:
            self.plot_x_dropdown.value = current
        elif checked:
            self.plot_x_dropdown.value = checked[0]

    def update_mode(self, change=None):
        mode = self.mode_dropdown.value
        self.x1_dropdown.disabled = mode == "n"
        self.x2_dropdown.disabled = mode != "two"
        self.plot_x_dropdown.disabled = mode != "n"
        for checkbox in self.x_checkboxes:
            checkbox.disabled = mode != "n"

    def display(self):
        display(HTML(f"<h3>{self.dataset_name} 선형회귀 확장 실습</h3>"))
        display(HTML(
            "<p>변수 선택 목록에는 수치형 데이터만 나타납니다. "
            "N개 입력 다중 회귀에서는 체크박스로 X 변수를 선택하고, 시각화 기준 X1을 지정합니다.</p>"
        ))
        if len(self.numeric_cols) < 2:
            display(HTML("<b>회귀 분석을 수행하려면 수치형 변수가 최소 2개 필요합니다.</b>"))
            return
        controls = VBox([
            self.mode_dropdown,
            HBox([self.x1_dropdown, self.x2_dropdown, self.y_dropdown]),
            self.n_box,
            self.run_button,
            self.output,
        ])
        display(controls)

    def validate_selection(self):
        mode = self.mode_dropdown.value
        y_col = self.y_dropdown.value
        plot_x = None

        if mode == "simple":
            x_cols = [self.x1_dropdown.value]
        elif mode == "two":
            x_cols = [self.x1_dropdown.value, self.x2_dropdown.value]
        else:
            x_cols = self.selected_checked_x()
            plot_x = self.plot_x_dropdown.value
            if len(x_cols) < 2:
                raise ValueError("N개 입력 다중 선형회귀는 X 변수를 2개 이상 체크해야 합니다.")
            if plot_x not in x_cols:
                raise ValueError("시각화 X1은 체크한 X 변수 중에서 선택해야 합니다.")

        if any(col is None for col in x_cols + [y_col]):
            raise ValueError("X와 Y 변수를 선택해 주세요.")
        if len(set(x_cols + [y_col])) != len(x_cols) + 1:
            raise ValueError("X와 Y는 서로 다른 변수를 선택해야 합니다.")
        return mode, x_cols, y_col, plot_x

    def build_model(self, x_cols, y_col):
        model_df = self.numeric_df[x_cols + [y_col]].dropna()
        if len(model_df) < 5:
            raise ValueError("결측치를 제거한 뒤 남는 데이터가 너무 적습니다. 다른 변수를 선택해 보세요.")

        X = model_df[x_cols]
        y = model_df[y_col]
        test_size = 0.25 if len(model_df) >= 12 else 0.2
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

        model = LinearRegression()
        model.fit(X_train, y_train)
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)
        pred_all = model.predict(X)

        metrics = pd.DataFrame(
            [
                {
                    "데이터": "학습 데이터",
                    "R²": r2_score(y_train, pred_train),
                    "MAE": mean_absolute_error(y_train, pred_train),
                    "RMSE": math.sqrt(mean_squared_error(y_train, pred_train)),
                    "데이터 수": len(y_train),
                },
                {
                    "데이터": "테스트 데이터",
                    "R²": r2_score(y_test, pred_test) if len(y_test) >= 2 else np.nan,
                    "MAE": mean_absolute_error(y_test, pred_test),
                    "RMSE": math.sqrt(mean_squared_error(y_test, pred_test)),
                    "데이터 수": len(y_test),
                },
            ]
        )
        metrics[["R²", "MAE", "RMSE"]] = metrics[["R²", "MAE", "RMSE"]].round(4)

        coef = pd.DataFrame(
            {
                "변수": ["절편"] + x_cols,
                "계수": [model.intercept_] + list(model.coef_),
            }
        )
        coef["계수"] = coef["계수"].round(6)

        result = model_df.copy()
        result["예측값"] = pred_all
        result["잔차(Y-예측값)"] = result[y_col] - result["예측값"]
        return model, metrics, coef, result

    def format_equation(self, y_col, coef, x_cols, model):
        equation = f"{y_col} = {coef.loc[0, '계수']}"
        for variable, value in zip(x_cols, model.coef_):
            equation += f" + ({value:.6f} × {variable})"
        return equation

    def run_regression(self, button=None):
        with self.output:
            clear_output(wait=True)
            try:
                mode, x_cols, y_col, plot_x = self.validate_selection()
                model, metrics, coef, result = self.build_model(x_cols, y_col)

                self.latest_mode = mode
                self.latest_x_cols = x_cols
                self.latest_y_col = y_col
                self.latest_plot_x = plot_x
                self.latest_model = model
                self.latest_result = result

                print("회귀식")
                print(self.format_equation(y_col, coef, x_cols, model))
                print("\n모델 성능")
                display(metrics)
                print("\n계수 해석")
                display(coef)
                print("\n시각화 준비 완료: 아래의 `app.show_visualization()` 코드 셀을 실행하면 Plotly 그래프가 별도 셀에 출력됩니다.")
            except Exception as error:
                print(f"오류: {error}")

    def simple_plot(self, result, x_col, y_col):
        sorted_result = result.sort_values(x_col)
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=result[x_col],
                y=result[y_col],
                mode="markers",
                name="실제 데이터",
                marker=dict(size=8, color="#2563eb", opacity=0.72),
                hovertemplate=f"{x_col}: %{{x}}<br>{y_col}: %{{y}}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=sorted_result[x_col],
                y=sorted_result["예측값"],
                mode="lines",
                name="회귀 추세선",
                line=dict(color="#dc2626", width=3),
                hovertemplate=f"{x_col}: %{{x}}<br>예측값: %{{y:.2f}}<extra></extra>",
            )
        )
        fig.update_layout(
            title=f"실제 데이터 산점도와 회귀 추세선: {x_col} → {y_col}",
            xaxis_title=x_col,
            yaxis_title=y_col,
            template=PLOTLY_TEMPLATE,
            height=540,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        )
        fig.show()

    def two_feature_3d_plot(self, result, x_cols, y_col, model):
        x1, x2 = x_cols
        fig = px.scatter_3d(
            result,
            x=x1,
            y=x2,
            z=y_col,
            color="잔차(Y-예측값)",
            color_continuous_scale="RdBu_r",
            hover_data={y_col: ":.2f", "예측값": ":.2f", "잔차(Y-예측값)": ":.2f"},
            title=f"3D 실제 데이터 산점도와 회귀 평면: {x1}, {x2} → {y_col}",
            template=PLOTLY_TEMPLATE,
            height=700,
        )

        x1_range = np.linspace(result[x1].min(), result[x1].max(), 28)
        x2_range = np.linspace(result[x2].min(), result[x2].max(), 28)
        grid_x1, grid_x2 = np.meshgrid(x1_range, x2_range)
        grid_pred = model.predict(
            pd.DataFrame({x1: grid_x1.ravel(), x2: grid_x2.ravel()})
        ).reshape(grid_x1.shape)

        fig.add_trace(
            go.Surface(
                x=grid_x1,
                y=grid_x2,
                z=grid_pred,
                name="회귀 평면",
                opacity=0.42,
                colorscale=[[0, "#f97316"], [1, "#f97316"]],
                showscale=False,
                hovertemplate=f"{x1}: %{{x:.2f}}<br>{x2}: %{{y:.2f}}<br>예측값: %{{z:.2f}}<extra></extra>",
            )
        )
        fig.update_layout(
            scene=dict(xaxis_title=x1, yaxis_title=x2, zaxis_title=y_col),
            coloraxis_colorbar=dict(title="잔차"),
        )
        fig.show()

    def n_feature_x1_plot(self, result, x_cols, y_col, plot_x, model):
        baseline = result[x_cols].mean()
        x_range = np.linspace(result[plot_x].min(), result[plot_x].max(), 120)
        grid = pd.DataFrame([baseline] * len(x_range))
        grid[plot_x] = x_range
        predicted_line = model.predict(grid[x_cols])

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=result[plot_x],
                y=result[y_col],
                mode="markers",
                name="실제 데이터",
                marker=dict(size=8, color="#2563eb", opacity=0.65),
                hovertemplate=f"{plot_x}: %{{x}}<br>{y_col}: %{{y}}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=x_range,
                y=predicted_line,
                mode="lines",
                name="X1 기준 예측 추세선",
                line=dict(color="#dc2626", width=3),
                hovertemplate=f"{plot_x}: %{{x:.2f}}<br>예측값: %{{y:.2f}}<extra></extra>",
            )
        )
        fixed_cols = [col for col in x_cols if col != plot_x]
        subtitle = ", ".join(fixed_cols[:4])
        if len(fixed_cols) > 4:
            subtitle += f" 외 {len(fixed_cols) - 4}개"
        fig.update_layout(
            title=f"N개 입력 회귀의 2D 해석: {plot_x} 변화와 {y_col} 예측<br><sup>나머지 X 변수는 평균값으로 고정: {subtitle}</sup>",
            xaxis_title=plot_x,
            yaxis_title=y_col,
            template=PLOTLY_TEMPLATE,
            height=540,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        )
        fig.show()

    def residual_plot(self, result, y_col):
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=result["예측값"],
            y=result["잔차(Y-예측값)"],
            mode="markers",
            name="잔차",
            marker=dict(color="#059669", size=8, opacity=0.72),
            hovertemplate="예측값: %{x:.2f}<br>잔차: %{y:.2f}<extra></extra>",
        ))
        fig.add_hline(y=0, line_dash="dash", line_color="#6b7280")
        fig.update_layout(
            title="잔차 확인: 예측값 주변에 잔차가 고르게 흩어져 있는가?",
            xaxis_title="예측값",
            yaxis_title=f"{y_col} - 예측값",
            template=PLOTLY_TEMPLATE,
            height=430,
        )
        fig.show()

    def show_visualization(self):
        if self.latest_result is None:
            print("먼저 위젯에서 변수를 선택하고 `회귀 분석 실행` 버튼을 눌러 주세요.")
            return

        result = self.latest_result
        x_cols = self.latest_x_cols
        y_col = self.latest_y_col

        if self.latest_mode == "simple":
            self.simple_plot(result, x_cols[0], y_col)
        elif self.latest_mode == "two":
            self.two_feature_3d_plot(result, x_cols, y_col, self.latest_model)
        else:
            self.n_feature_x1_plot(result, x_cols, y_col, self.latest_plot_x, self.latest_model)

        self.residual_plot(result, y_col)
        print("해석 관점")
        print("- R²가 높다고 해서 반드시 원인과 결과가 증명되는 것은 아닙니다.")
        print("- 계수의 부호와 크기는 다른 선택 변수와 데이터 범위 안에서 해석해야 합니다.")
        print("- 잔차가 특정 방향으로 몰리면 직선 모델보다 다른 설명 방식이 필요할 수 있습니다.")

In [ ]:
app = ExtendedLinearRegressionColabApp(df, dataset_name=globals().get("dataset_name", "춘천 닭갈비 상점별 최고 이용자수"))
app.display()

## 6. 시각화 결과 확인
위젯에서 `회귀 분석 실행` 버튼을 누른 뒤 아래 셀을 실행합니다. Plotly 그래프는 ipywidgets 출력 영역이 아니라 이 코드 셀의 일반 출력 영역에 표시됩니다.

In [ ]:
app.show_visualization()

## 사용 팁
- 단순 선형회귀에서는 산점도와 빨간 추세선을 보며 한 변수와 결과 변수의 관계를 확인합니다.
- X가 2개인 다중 선형회귀에서는 3D 산점도와 회귀 평면을 회전하며 확인합니다.
- X가 N개인 다중 선형회귀에서는 전체 차원을 한 번에 볼 수 없으므로, 지정한 시각화 X1만 변화시키고 나머지 X는 평균값으로 고정해 2D 추세를 확인합니다.
- 수치형 변수만 사용할 수 있으므로 상호명, 날짜, 업종처럼 문자로 읽힌 컬럼은 변수 선택 목록에서 제외됩니다.
- 회귀 결과는 데이터 안의 경향을 설명하는 도구입니다. 지역, 시기, 표본 수, 누락 변수까지 함께 생각해야 합니다.